#### Extract youtube transcripts

Install library

In [1]:
# %pip install yt-dlp youtube-transcript-api

# https://github.com/jdepoix/youtube-transcript-api

In [2]:
import time
from datetime import date

import pandas as pd

from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
)

print("imports OK")

imports OK


Step 1: Parameters in one place

In [13]:
youtube_url="https://www.youtube.com/"
channel_name="@cuny"

start_date= date(2026, 11, 1)
end_date= date(2026, 11, 10) ## date.today()
lang=['en']
sleep=1.0

url=f"{youtube_url}{channel_name}"

print(url, end_date-start_date)

https://www.youtube.com/@cuny 9 days, 0:00:00


Step 3: Date helper

In [14]:
# Convert "20250314" -> date(2025, 3, 14).

def parse(raw):
    return date(int(raw[:4]), int(raw[4:6]), int(raw[6:]))

# test it
print(parse("20250314"))

2025-03-14


Step 4: Get the list of videos IDs

In [15]:
video_url = url.rstrip("/") + "/videos"

with YoutubeDL({"extract_flat": "in_playlist", "quiet": True}) as ydl:
    playlist = ydl.extract_info(video_url, download=False)

ids = [e["id"] for e in playlist["entries"] if e.get("id")]

print("Total videos:", len(ids))
print("First 5 IDs:", ids[:5])

Total videos: 1924
First 5 IDs: ['K21Iams_JTM', 'RuCSMaKKNA4', 'XZrw_EFdeqI', 'EETV4MKbIO8', '9U804PTnoJw']


In [16]:
f"https://www.youtube.com/watch?v={ids[0]}"

'https://www.youtube.com/watch?v=K21Iams_JTM'

Step 5: Look at 1 video’s info before looping

In [17]:
with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
    info = ydl.extract_info(
        f"https://www.youtube.com/watch?v={ids[0]}",
        download=False,
        process=False,
    )

print(info["title"])
print(info["upload_date"], "parsed to", parse(info["upload_date"]))

UACD History of Diversity at CUNY 2026
20261106 parsed to 2026-11-06


In [18]:
fields = ["view_count", "like_count", "comment_count", "duration",
          "channel_follower_count", "categories", "tags"]

for f in fields:
    print(f, ":", info.get(f))

view_count : 107
like_count : 4
comment_count : None
duration : 5413
channel_follower_count : 15300
categories : ['People & Blogs']
tags : []


Step 6: Filter by date

In [19]:
found = []

with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
    for vid in ids:
        try:
            info = ydl.extract_info(
                f"https://www.youtube.com/watch?v={vid}",
                download=False,
                process=False,
            )
        except DownloadError as err:
            print("skip", vid, err)
            continue

        if not info.get("upload_date"):
            continue
        d = parse(info["upload_date"])

        if d > end_date:
            print("too new: ", d, info["title"])
            continue
        if d < start_date:
            print("too old: ", d, "-> stopping")
            break

        found.append({
            "video_id": vid,
            "date": d,
            "title": info["title"],
            "views": info.get("view_count"),
            "likes": info.get("like_count"),
            "comments": info.get("comment_count"),
            "duration_sec": info.get("duration"),
            "subscribers": info.get("channel_follower_count")            })
        print("iin range:", d, info["title"])
        time.sleep(sleep)

print("Videos in range:", len(found))

iin range: 2026-11-06 UACD History of Diversity at CUNY 2026
iin range: 2026-11-04 CUNY Grad with HOT 97 Energy 🔥
iin range: 2026-11-04 Navigating Immigration Changes, Updates for the CUNY Community
too old:  2026-10-30 -> stopping
Videos in range: 3


In [20]:
ytt = YouTubeTranscriptApi()
test = ytt.fetch(found[0]["video_id"], languages=lang)
print("Segments:", len(test))
print("Auto-generated?", test.is_generated)
print(test[0])
print(test[1])

Segments: 2044
Auto-generated? True
FetchedTranscriptSnippet(text='started. Thank you so much. Everyone', start=6.4, duration=6.48)
FetchedTranscriptSnippet(text='could have a seat. Good afternoon', start=10.0, duration=5.28)


In [21]:
segments = []
videos = []

for v in found:
    try:
        t = ytt.fetch(v["video_id"], languages=lang)
    except (TranscriptsDisabled, NoTranscriptFound):
        print("no transcript:", v["title"])
        continue
    except Exception as err:
        print("error:", v["video_id"], err)
        continue

    for s in t:
        segments.append({
            **v,
            "start": s.start,
            "duration": s.duration,
            "text": s.text,
            "url": f"https://youtu.be/{v['video_id']}?t={int(s.start)}",
        })

    videos.append({
        **v,
        "is_generated": t.is_generated,
        "text": " ".join(s.text for s in t),
    })
    print("got:", v["date"], v["title"])
    time.sleep(sleep)

print("\nSegments:", len(segments), "| Videos:", len(videos))

got: 2026-11-06 UACD History of Diversity at CUNY 2026
got: 2026-11-04 CUNY Grad with HOT 97 Energy 🔥
got: 2026-11-04 Navigating Immigration Changes, Updates for the CUNY Community

Segments: 4031 | Videos: 3


In [22]:
seg_df = pd.DataFrame(segments)
vid_df = pd.DataFrame(videos)

vid_df.head()

,video_id,date,title,views,likes,comments,duration_sec,subscribers,is_generated,text
0,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,True,started. Thank you so much. Everyone could hav...
1,RuCSMaKKNA4,2026-11-04,CUNY Grad with HOT 97 Energy 🔥,74,5,1.0,45,15300,True,I graduated from Bronx Community College with ...
2,XZrw_EFdeqI,2026-11-04,"Navigating Immigration Changes, Updates for th...",66,1,NaN,4488,15300,True,Hello everyone and welcome back to the monthly...


In [24]:
word = "diversity"   # change this
hits = seg_df[seg_df["text"].str.contains(word, case=False)]

print(len(hits), "mentions")
hits.head(10)

52 mentions


,video_id,date,title,views,likes,comments,duration_sec,subscribers,start,duration,text,url
8,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,24.960,5.440,diversity's hosting of a panel event on,https://youtu.be/K21Iams_JTM?t=24
9,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,27.519,5.200,the history of diversity at CUNI. My,https://youtu.be/K21Iams_JTM?t=27
21,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,54.879,5.121,event. Diversity is at the core of,https://youtu.be/K21Iams_JTM?t=54
53,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,126.719,4.001,"embrace of the values of diversity,",https://youtu.be/K21Iams_JTM?t=126
57,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,133.680,6.000,diversity at CUNI is the result of a,https://youtu.be/K21Iams_JTM?t=133
102,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,236.480,4.720,diversity at CUNI which would indeed be,https://youtu.be/K21Iams_JTM?t=236
111,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,256.239,3.761,diversity conference and engage with,https://youtu.be/K21Iams_JTM?t=256
203,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,455.520,4.350,diversity at CUNI.,https://youtu.be/K21Iams_JTM?t=455
212,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,485.199,6.801,"inaugural history of diversity, a cutuny",https://youtu.be/K21Iams_JTM?t=485
215,K21Iams_JTM,2026-11-06,UACD History of Diversity at CUNY 2026,107,4,NaN,5413,15300,494.479,5.921,diversity. I um am going to shorten my,https://youtu.be/K21Iams_JTM?t=494


How to do this using YouTube API:

1. Go to the Google Cloud Console (console.cloud.google.com) and create a project.
2. Under APIs & Services → Library, search for “YouTube Data API v3” and click Enable.
3. Under APIs & Services → Credentials, click Create credentials → API key.